In [9]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START
from langgraph.types import Command
from langgraph.constants import END
from dotenv import load_dotenv
from langchain_typesafe import Choice, Noul, Score, TypeSafeClassifier

load_dotenv()

classifier = TypeSafeClassifier()


# 每种意图的场景
INTENT_CLASIFY={
    "product": "产品功能、使用方法、规则参数、故障等产品相关问题",
    "service": "售后服务、退换货、维修等售后问题",
    "policy": "政策法规、合同条款、隐私政策等法律问题",
    "human": "其他无法通过关键词或模型识别的问题，需要人工处理或者明确要求人工处理的问题。"
}

# 每种意图对应的处理节点和描述
INTENT_TO_NODE={
    "product":("product", "处理产品问题"),
    "service":("service", "处理售后问题"),
    "policy":("policy", "处理政策问题"),
    "human":("human", "人工处理问题")
}

class State(TypedDict):
    inputs: str
    outputs: str

def product(state: State) -> State:
    return {"outputs": state["inputs"]+"产品！"}

def service(state: State) -> State:
    return {"outputs": state["inputs"]+"售后！"}

def policy(state: State) -> State:
    return {"outputs": state["inputs"]+"政策！"}
def human(state: State) -> State:
    return {"outputs": state["inputs"]+"人工处理！"}

def sum_all(state: State) -> State:
    return {"outputs": state["inputs"]}

def router(state: State) -> Command[Literal["product", "service","policy", "human"]]:
    input_content = state["inputs"]
    destination = None
    change_state = None
    
    #意图识别【关键词|微调模型|JEV】
    # 1.先通过关键词识别
    if "产品" in input_content:
        destination = "product"
        change_state = {"inputs": "处理产品问题"}
    elif "售后" in input_content:
        destination = "service"
        change_state = {"inputs": "处理售后问题"}
    elif "政策" in input_content:
        destination = "policy"
        change_state = {"inputs": "处理政策问题"}
    else:
        # 2.使用jev模型进行意图识别
        response = classifier.invoke(
            {
                "state": input_content ,
                "questions": {
                    "intent": Choice(
                        instructions="判断用户问题的意图类别。",
                        criteria=INTENT_CLASIFY,
                    ),
                },
            }
        )
        """
        ClassifierResponse(
            model='typesafe/jev-1.14-20261014', 
            answers={
                'intent': ChoiceAnswer(
                    type='choice', 
                    choice='product', 
                    probabilities={
                        'product': 0.55, 
                        'service': 0.45, 
                        'policy': 0.0, 
                        'human': 0.0
                    }, 
                    confidence=0.4
                )
            }, 
            usage=Usage(input_tokens=440, output_tokens=45), 
            request_id=None
        )
        """
        answer = response.answers["intent"]
        destination, next_input =INTENT_TO_NODE.get(answer.choice,INTENT_TO_NODE["human"])
        change_state = {"inputs": next_input}
   
    # goto: 下一步节点
    # update: 更新状态
    return Command(goto=destination, update=change_state)

builder = StateGraph(State)
builder.add_node("router", router)
builder.add_node("product", product)
builder.add_node("service",service)
builder.add_node("policy",policy)
builder.add_node("human",human)
builder.add_node("sum_all",sum_all)

builder.add_edge(START, "router")
builder.add_edge("product", "sum_all")
builder.add_edge("service", "sum_all")
builder.add_edge("policy", "sum_all")
builder.add_edge("human", "sum_all")
builder.add_edge("sum_all", END)
graph = builder.compile()

In [10]:
res = graph.invoke({"inputs": "我之前购买的手机现在不能开机！"})
print(res)

{'inputs': '处理售后问题', 'outputs': '处理售后问题'}


In [11]:
res = graph.invoke({"inputs": "我之前购买的手机我不知道怎么用！"})
print(res)

{'inputs': '处理产品问题', 'outputs': '处理产品问题'}


In [12]:
res = graph.invoke({"inputs": "我想退货，请问你们这退货的规则是怎么样的？"})
print(res)

{'inputs': '处理售后问题', 'outputs': '处理售后问题'}


In [13]:
res = graph.invoke({"inputs": "我想换新的手机，你这有法律支持吗？"})
print(res)

{'inputs': '处理政策问题', 'outputs': '处理政策问题'}


In [14]:
res = graph.invoke({"inputs": "how are you ?"})
print(res)

{'inputs': '人工处理问题', 'outputs': '人工处理问题'}
